# RAGLedger — reproducible Google Colab GPU workflow

This notebook is the executable Colab companion to `docs/COLAB_RUNBOOK.md`. It is
one linear workflow: GPU check → pinned clone → Drive → install → model caches →
dataset subsets → functional preflight → **integration pilot** → optional
**separate research run** → resume → results export.

**Scope rule.** The pilot uses Qwen2.5-7B via Ollama q4_K_M as an explicitly
different model from the research configuration. It proves integration only.
The research configuration remains Contriever + DeBERTa NLI + Llama-3.1-8B-Instruct
(unquantised bfloat16, vLLM). The two are never equivalent and their outputs are
stored in different directories. Answer-level ASR remains **NOT MEASURED**.

Before running: **Runtime → Change runtime type → GPU**. The notebook does not
download the full NQ corpus or a large generator without an explicit execution
flag. It was not run in Colab during this repository update; see the static
validation report at the end.


## 1. Runtime selection and GPU verification

In [ ]:
import json, os, pathlib, shutil, sqlite3, subprocess, sys, tarfile, time, urllib.request

import torch

print("nvidia-smi:")
subprocess.run(["nvidia-smi", "-L"], check=False)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Runtime → Change runtime type → GPU, then rerun from this cell.")
props = torch.cuda.get_device_properties(0)
FREE_VRAM_GB = torch.cuda.mem_get_info(0)[0] / 1e9
HARDWARE = {
    "gpu": props.name,
    "vram_gb": round(props.total_memory / 1e9, 2),
    "free_vram_gb": round(FREE_VRAM_GB, 2),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "python": sys.version.split()[0],
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 2),
    "disk_free_gb": round(shutil.disk_usage("/content").free / 1e9, 2),
}
print(json.dumps(HARDWARE, indent=2))
assert torch.cuda.is_available()


## 2. Clone the correct repository and pin the commit

In [ ]:
REPO_URL = "https://github.com/athishss/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git"
PINNED_COMMIT = "2bac018997a80ba1cf6d8cdc49e8ec20efc4660d"
REPO = pathlib.Path("/content/RAGLedger")

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", PINNED_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", PINNED_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == PINNED_COMMIT, (actual, PINNED_COMMIT)
print("repository:", REPO_URL)
print("checked-out SHA:", actual)
os.environ["RAGLEDGER_COMMIT"] = actual


## 3. Mount Google Drive and create persistent directories

Temporary model caches stay in `/content/hf_cache`. Datasets, prepared subsets,
config snapshots, indexes, the SQLite ledger, checkpoints, manifests, metrics,
logs and exports live under `MyDrive/RAGLedger/`. A runtime reset does not lose
those results. Each run uses a unique directory, and resume is forbidden across
different signatures.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = pathlib.Path("/content/drive/MyDrive/RAGLedger")
for name in ("datasets/raw", "datasets/pilot", "datasets/research", "configs",
             "runs", "preflight", "exports", "logs"):
    (DRIVE / name).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
pathlib.Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("persistent root:", DRIVE)
print("HF cache (temporary):", os.environ["HF_HOME"])


## 4. Install dependencies and verify the environment

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all,data]", "matplotlib"], check=True)
import faiss, transformers, pyarrow
import torch as _torch
print("torch", _torch.__version__, "| cuda", _torch.version.cuda)
print("transformers", transformers.__version__, "| pyarrow", pyarrow.__version__)
print("faiss", faiss.__version__, "| GPU FAISS:", hasattr(faiss, "StandardGpuResources"))
assert not hasattr(faiss, "StandardGpuResources"), "This environment unexpectedly has GPU FAISS; update the hardware note."
print("FAISS index execution: CPU (faiss-cpu). This is not GPU-accelerated indexing.")


## 5. Cache the embedding and NLI weights (user-controlled)

Weights are pinned to exact Hugging Face commit SHAs before the functional
preflight enables offline mode. This means the doctor cannot silently download a
different revision. The generator for the pilot is pulled separately below.


In [ ]:
from huggingface_hub import snapshot_download

EMBED_ID = "facebook/contriever"
EMBED_REV = "2bd46a25019aeea091fd42d1f0fd4801675cf699"
NLI_ID = "cross-encoder/nli-deberta-v3-base"
NLI_REV = "6c749ce3425cd33b46d187e45b92bbf96ee12ec7"
LOCK = {"embedding": {"model": EMBED_ID, "revision": EMBED_REV},
        "nli": {"model": NLI_ID, "revision": NLI_REV}}
for item in LOCK.values():
    path = snapshot_download(item["model"], revision=item["revision"])
    print(item["model"], item["revision"], "->", path)
(DRIVE / "configs" / "model_pins.json").write_text(json.dumps(LOCK, indent=2))


## 6. Pull the explicitly alternate pilot generator

This is **Qwen2.5-7B via Ollama q4_K_M**, not the research Llama model. The
notebook records the pulled manifest digest and actual quantisation, then passes
those exact values to both preflight and runner.


In [ ]:
if not shutil.which("zstd"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        return True
    except Exception:
        return False
if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "ab"), stderr=subprocess.STDOUT)
for _ in range(30):
    if ollama_up():
        break
    time.sleep(2)
assert ollama_up(), "Ollama API did not start; inspect /tmp/ollama.log"
PILOT_MODEL = "qwen2.5:7b"
subprocess.run(["ollama", "pull", PILOT_MODEL], check=True)
tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))
entry = next(m for m in tags["models"] if m["name"] == PILOT_MODEL)
QWEN_DIGEST = entry["digest"]
request = urllib.request.Request("http://127.0.0.1:11434/api/show",
    data=json.dumps({"name": PILOT_MODEL}).encode(), headers={"Content-Type": "application/json"})
shown = json.load(urllib.request.urlopen(request))
QWEN_QUANT = shown.get("details", {}).get("quantization_level", "UNSPECIFIED")
print("pilot generator:", PILOT_MODEL, QWEN_DIGEST, QWEN_QUANT)
assert QWEN_QUANT != "UNSPECIFIED"


## 7. Download raw NQ data and build separate pilot/research subsets

The full raw BEIR file is downloaded once. Gold passages are always retained;
the fill is deterministic. The pilot gets 5,000 passages / 20 queries. The
research preparation is gated and gets 200,000 / 500.


In [ ]:
RUN_DATASET_DOWNLOAD = True
RUN_RESEARCH = False
RAW = DRIVE / "datasets" / "raw"
if RUN_DATASET_DOWNLOAD and not (RAW / "corpus.parquet").exists():
    subprocess.run([sys.executable, str(REPO / "scripts/download_data.py"), "--dataset", "nq",
                    "--out", str(RAW), "--queries", "500"], cwd=REPO, check=True)
assert (RAW / "corpus.parquet").exists(), "Raw corpus missing; enable RUN_DATASET_DOWNLOAD."
print("raw corpus bytes:", (RAW / "corpus.parquet").stat().st_size)


In [ ]:
import importlib.util, random
spec = importlib.util.spec_from_file_location("download_data", REPO / "scripts/download_data.py")
dd = importlib.util.module_from_spec(spec); spec.loader.exec_module(dd)
gold = dd.read_qrels(RAW / "qrels.tsv")

def prepare(outdir, passages, queries, seed=20260921):
    outdir.mkdir(parents=True, exist_ok=True)
    chosen = sorted(gold); random.Random(seed).shuffle(chosen); chosen = chosen[:queries]
    wanted = set().union(*(gold[q] for q in chosen))
    dd.build_subset(RAW / "corpus.parquet", wanted, passages, outdir / "corpus_subset.parquet", seed)
    dd.write_queries_subset(RAW / "queries.parquet", chosen, outdir / "queries_subset.jsonl")
    shutil.copy2(RAW / "qrels.tsv", outdir / "qrels.tsv")
    return outdir / "corpus_subset.parquet", outdir / "queries_subset.jsonl", outdir / "qrels.tsv"

PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS = prepare(DRIVE / "datasets/pilot", 5000, 20)
print("pilot ready:", PILOT_CORPUS)
if RUN_RESEARCH:
    RESEARCH_CORPUS, RESEARCH_QUERIES, RESEARCH_QRELS = prepare(DRIVE / "datasets/research", 200000, 500)
    print("research ready:", RESEARCH_CORPUS)
else:
    print("research data not prepared (RUN_RESEARCH=False)")
assert sum(1 for _ in open(PILOT_QUERIES)) == 20


## 8. Functional preflight for the pilot configuration

The same config and **same `--set` overrides** are passed here and to the runner.
Preflight runs functional embedding, generation and NLI inference with offline
HF flags, checks the endpoint and persistent SQLite ledger, and must return
`READY` / `PROTECTED`.


In [ ]:
PILOT_CONFIG = REPO / "config/colab_pilot_qwen.yaml"
PILOT_RUN = DRIVE / "runs" / "pilot_qwen25_7b_q4"
PILOT_OVERRIDES = [f"embedding.revision={EMBED_REV}", f"security.nli_revision={NLI_REV}",
                   f"generation.revision={QWEN_DIGEST}", f"generation.quantization={QWEN_QUANT}",
                   f"storage.root={PILOT_RUN}"]

def preflight(config, corpus, queries, qrels, overrides, label):
    cmd = [sys.executable, str(REPO / "scripts/check_research_environment.py"),
           "--config", str(config), "--functional", "--json",
           "--corpus", str(corpus), "--queries", str(queries), "--qrels", str(qrels)]
    for item in overrides:
        cmd += ["--set", item]
    proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    (DRIVE / "preflight" / f"{label}.log").write_text(proc.stdout + proc.stderr)
    payload = json.loads(proc.stdout[proc.stdout.find("{"):])
    (DRIVE / "preflight" / f"{label}.json").write_text(json.dumps(payload, indent=2))
    print(label, payload["status"], payload["status_reason"])
    if payload["status"] != "READY":
        print(json.dumps(payload.get("failures", []), indent=2))
        raise RuntimeError(f"preflight is {payload['status']}, not READY")
    generation = payload["config"]["generation_model"]
    print("checked model:", generation, "| backend:", payload["config"]["generation_backend"])
    return payload

PILOT_PREFLIGHT = preflight(PILOT_CONFIG, PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS,
                            PILOT_OVERRIDES, "pilot")
ps = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/ps"))
running = next((m for m in ps.get("models", []) if m.get("name") == PILOT_MODEL), None)
assert running and int(running.get("size_vram", 0)) > 0, "Ollama model is not GPU-resident"
print("generator GPU residency verified: size_vram =", running["size_vram"])


## 9. Integration pilot (5,000 passages, 20-query pool, 12 Zipf steps)

This is a component/integration pilot, not a research result. The 12 stream steps
sample with replacement; the manifest records the 20-query pool, 12 steps and the
actual number of distinct queries.


In [ ]:
def run_stream(config, corpus, queries, qrels, out, purpose, steps, targets, limit, overrides):
    if (out / "metrics.json").exists():
        print("completed run already present; not overwriting:", out)
        return json.loads((out / "run_manifest.json").read_text())
    cmd = [sys.executable, str(REPO / "scripts/run_trust_stream.py"),
           "--config", str(config), "--corpus", str(corpus), "--queries", str(queries),
           "--qrels", str(qrels), "--out", str(out), "--run-purpose", purpose,
           "--steps", str(steps), "--targets", str(targets)]
    if limit is not None:
        cmd += ["--limit", str(limit)]
    for item in overrides:
        cmd += ["--set", item]
    if (out / "run_manifest.json").exists() and not (out / "metrics.json").exists():
        cmd.append("--resume")
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"stream failed with exit {proc.returncode}")
    return json.loads((out / "run_manifest.json").read_text())

PILOT_MANIFEST = run_stream(PILOT_CONFIG, PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS,
                            PILOT_RUN, "pilot", 12, 2, 5000, PILOT_OVERRIDES)
assert PILOT_MANIFEST["run_purpose"] == "pilot"
assert PILOT_MANIFEST["real_model"] is True
assert PILOT_MANIFEST["models"]["equivalence_claim"] is False
assert PILOT_MANIFEST["models"]["generation"]["model_id"] == PILOT_MODEL
assert PILOT_MANIFEST["query_pool_size"] == 20
assert PILOT_MANIFEST["stream_steps_completed"] == 12
assert PILOT_MANIFEST["models"]["verifier"]["backend"] == "nli"
print(json.dumps({k: PILOT_MANIFEST[k] for k in ("state", "run_purpose", "query_pool_size",
      "stream_steps_completed", "distinct_queries_evaluated", "research_findings_allowed")}, indent=2))


## 10. Separate research execution (gated)

Set `RUN_RESEARCH=True` only on a GPU that can serve the pinned Llama-3.1-8B
research model. The code installs/starts vLLM, runs the functional preflight on
the **same** research config and overrides, then evaluates 500 unique queries
without replacement. This still is **not** the full attack × baseline matrix and
answer-level ASR remains NOT MEASURED.

The vLLM weight download is user-controlled by this flag. A free T4 is not a
supported research host for this model.


In [ ]:
if not RUN_RESEARCH:
    print("Research run skipped: RUN_RESEARCH=False. This is intentional; prepare the 200K dataset and use an adequate GPU first.")
else:
    LLAMA_ID = "meta-llama/Llama-3.1-8B-Instruct"
    LLAMA_REV = "a09a35458c702b33eeacc393d103063234e8bc28"
    RESEARCH_CONFIG = REPO / "config/nq_gpu.yaml"
    RESEARCH_RUN = DRIVE / "runs" / "research_llama31_8b_bf16"
    RESEARCH_OVERRIDES = [f"embedding.revision={EMBED_REV}", f"security.nli_revision={NLI_REV}",
                          f"generation.revision={LLAMA_REV}", "generation.backend=openai",
                          f"generation.model_name={LLAMA_ID}", "generation.precision=bfloat16",
                          "generation.base_url=http://127.0.0.1:8000/v1",
                          f"storage.root={RESEARCH_RUN}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
    vllm_log = open(DRIVE / "logs" / "vllm.log", "ab")
    subprocess.Popen(["vllm", "serve", LLAMA_ID, "--revision", LLAMA_REV,
                      "--port", "8000", "--dtype", "bfloat16", "--gpu-memory-utilization", "0.55"],
                     stdout=vllm_log, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2).read(); break
        except Exception: time.sleep(5)
    RESEARCH_PREFLIGHT = preflight(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                                   RESEARCH_QRELS, RESEARCH_OVERRIDES, "research")
    RESEARCH_MANIFEST = run_stream(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                                   RESEARCH_QRELS, RESEARCH_RUN, "research", 500, 5, None,
                                   RESEARCH_OVERRIDES)
    assert RESEARCH_MANIFEST["query_pool_size"] == 500
    assert RESEARCH_MANIFEST["stream_steps_completed"] == 500
    assert RESEARCH_MANIFEST["distinct_queries_evaluated"] == 500
    assert RESEARCH_MANIFEST["research_findings_allowed"] is True
    assert RESEARCH_MANIFEST["research_evaluation"]["full_attack_baseline_matrix"] is False
    print("Research trust-stream run completed; this is not the full C6 benchmark.")


In [ ]:
# Optional separate Zipf trust-lifecycle experiment on the research corpus.
# It is NOT a 500-unique-query evaluation. It is kept in its own directory.
RUN_RESEARCH_ZIPF = False
if RUN_RESEARCH_ZIPF and RUN_RESEARCH:
    LIFE_RUN = DRIVE / "runs" / "research_zipf_lifecycle"
    LIFE_MANIFEST = run_stream(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                               RESEARCH_QRELS, LIFE_RUN, "research_lifecycle", 500, 5, None,
                               RESEARCH_OVERRIDES)
    print("Zipf lifecycle steps:", LIFE_MANIFEST["stream_steps_completed"],
          "distinct queries:", LIFE_MANIFEST["distinct_queries_evaluated"])
else:
    print("Zipf research-lifecycle run skipped (separate from the 500-unique-query research run).")


## 11. Resume after a runtime disconnection

Re-running the pilot cell resumes only when `checkpoint.json`, the saved index,
the SQLite ledger and the full run signature still match. Completed steps are
read from `stream.jsonl`. If the disconnect happened *inside* a step, resume is
refused rather than mixing partial trust effects. This cell only inspects state;
the run cell performs the safe resume.


In [ ]:
def inspect_run(path):
    ckpt = path / "checkpoint.json"
    if not ckpt.exists():
        print(path.name, "has no checkpoint")
        return
    c = json.loads(ckpt.read_text())
    print(path, "phase=", c.get("phase"), "completed=", c.get("completed_steps"),
          "in_flight=", c.get("in_flight_step"))
inspect_run(PILOT_RUN)
if RUN_RESEARCH and (DRIVE / "runs/research_llama31_8b_bf16").exists():
    inspect_run(DRIVE / "runs/research_llama31_8b_bf16")


## 12. Metrics, manifest and visualization

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show_results(path):
    m = json.loads((path / "metrics.json").read_text())
    manifest = json.loads((path / "run_manifest.json").read_text())
    print("run:", manifest["run_purpose"], "| state:", manifest["state"])
    print("models:", json.dumps(manifest["models"], indent=2))
    for key in ("steps", "query_pool_size", "query_sampling", "distinct_queries_evaluated",
                "poison_retrieved_rate", "poison_exposure_rate", "poison_citation_rate",
                "poison_quarantined", "clean_false_quarantine", "answer_level_asr"):
        if key in m: print(f"{key:>28}: {m[key]}")
    print("answer-level ASR:", m.get("answer_level_asr", "NOT MEASURED"))
    for png in sorted((path / "figures").glob("*.png"))[:4]:
        display(Image(filename=str(png)))
show_results(PILOT_RUN)


## 13. Export and preserve final results

The export includes the manifest, metrics, stream, history, config snapshot,
checkpoint and a consistent SQLite backup. Large FAISS indexes remain in the run
directory on Drive so resumed runs can reuse them without duplicating them in an
archive.


In [ ]:
def export_run(path, label):
    target = DRIVE / "exports" / label
    target.mkdir(parents=True, exist_ok=True)
    for name in ("run_manifest.json", "metrics.json", "stream.jsonl", "trust_history.csv",
                 "checkpoint.json"):
        source = path / name
        if source.exists(): shutil.copy2(source, target / name)
    if (path / "trust.sqlite3").exists():
        src = sqlite3.connect(path / "trust.sqlite3"); dst = sqlite3.connect(target / "trust.sqlite3")
        with dst: src.backup(dst)
        dst.close(); src.close()
    manifest = json.loads((path / "run_manifest.json").read_text())
    (target / "config_snapshot.json").write_text(json.dumps(manifest["config"], indent=2))
    if (path / "figures").exists():
        shutil.copytree(path / "figures", target / "figures", dirs_exist_ok=True)
    archive = DRIVE / "exports" / f"{label}_{time.strftime('%Y%m%d-%H%M%S')}"
    shutil.make_archive(str(archive), "gzip", root_dir=target)
    print("exported:", archive.with_suffix(".tar.gz"))
export_run(PILOT_RUN, "pilot_qwen25_7b_q4")
if RUN_RESEARCH and (DRIVE / "runs/research_llama31_8b_bf16/metrics.json").exists():
    export_run(DRIVE / "runs/research_llama31_8b_bf16", "research_llama31_8b_bf16")


## Validation status and remaining work

Executed in this repository update: notebook JSON validation, Python AST parsing
of every code cell, config/preflight/runner unit tests, and a local offline mini
run. **Not executed:** this notebook in Google Colab; no GPU, NLI checkpoint,
Ollama model or NQ download was available locally. Claims of Colab success are
therefore not made.

Remaining before genuine research: run this notebook on a Colab GPU; complete the
pilot; confirm the manifest records real model IDs/revisions and `PROTECTED`;
then use an adequate GPU, prepare 200K/500, start pinned Llama, and run the
separate research cell. The full attack × baseline benchmark, ASR criteria,
baselines and confidence intervals remain Person C work in `docs/PERSON_C_TODO.md`.
